In [1]:
import pyrealsense2 as rs
import json, cv2, numpy as np, os

# =====================================================
# CHANGE THIS ONLY
# =====================================================
FFB_ID = 17   # ← change to 10, 11, 12, etc.

# =====================================================
# PATHS (AUTO-UPDATED)
# =====================================================
bag_path = rf"C:\Users\admin\Documents\Vicki\School stuff\SEGP\FFB Samples\FFB{FFB_ID}\ffb{FFB_ID}Depth_3D.bag"
out_dir  = rf"sample_{FFB_ID:03d}"
os.makedirs(out_dir, exist_ok=True)

print(f"Processing FFB {FFB_ID}")
print("Bag file:", bag_path)
print("Output dir:", out_dir)

# =====================================================
# REALSENSE PIPELINE
# =====================================================
pipe = rs.pipeline()
cfg = rs.config()
cfg.enable_device_from_file(bag_path, repeat_playback=False)
cfg.enable_stream(rs.stream.color)
cfg.enable_stream(rs.stream.depth)

profile = pipe.start(cfg)

align = rs.align(rs.stream.color)
depth_sensor = profile.get_device().first_depth_sensor()
depth_scale = depth_sensor.get_depth_scale()

color_stream = profile.get_stream(rs.stream.color).as_video_stream_profile()
intr = color_stream.get_intrinsics()

# =====================================================
# SAVE INTRINSICS
# =====================================================
intr_path = os.path.join(out_dir, "intrinsics.json")
with open(intr_path, "w") as f:
    json.dump({
        "fx": intr.fx,
        "fy": intr.fy,
        "cx": intr.ppx,
        "cy": intr.ppy,
        "width": intr.width,
        "height": intr.height,
        "depth_scale": depth_scale
    }, f, indent=2)

print("Saved intrinsics to:", intr_path)

# =====================================================
# EXTRACT FRAMES
# =====================================================
i = 0
try:
    while True:
        frames = pipe.wait_for_frames()
        frames = align.process(frames)

        depth = frames.get_depth_frame()
        color = frames.get_color_frame()
        if not depth or not color:
            continue

        i += 1
        rgb = np.asanyarray(color.get_data())
        dep = np.asanyarray(depth.get_data())  # uint16

        cv2.imwrite(os.path.join(out_dir, f"rgb_{i:04d}.png"),
                    cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
        cv2.imwrite(os.path.join(out_dir, f"depth_{i:04d}.png"), dep)

except Exception as e:
    print("Stopped:", e)

finally:
    pipe.stop()
    print(f"Done. Extracted {i} frames.")


Processing FFB 17
Bag file: C:\Users\admin\Documents\Vicki\School stuff\SEGP\FFB Samples\FFB17\ffb17Depth_3D.bag
Output dir: sample_017
Saved intrinsics to: sample_017\intrinsics.json
Stopped: Frame didn't arrive within 5000
Done. Extracted 65 frames.


In [2]:
import cv2, numpy as np, json, glob, os

# =====================================================
# MUST MATCH THE FIRST CELL
# =====================================================
FFB_ID = 17   # ← same value as above

# =====================================================
# PATHS (AUTO)
# =====================================================
frames_dir = f"sample_{FFB_ID:03d}"

# --- load intrinsics (unchanged logic)
with open(f"{frames_dir}/intrinsics.json", "r") as f:
    meta = json.load(f)

scale = meta["depth_scale"]

# --- robust depth loading (LOGIC SAME: still one depth image)
depth_files = sorted(glob.glob(f"{frames_dir}/depth_*.png"))
assert len(depth_files) > 0, f"No depth images found in {frames_dir}"

D = cv2.imread(depth_files[0], cv2.IMREAD_UNCHANGED)
assert D is not None, f"Failed to load {depth_files[0]}"

# =====================================================
# ORIGINAL LOGIC (UNCHANGED)
# =====================================================
valid = D > 0

print(f"FFB {FFB_ID}")
print("depth_scale:", scale)
print("valid %:", 100 * valid.mean())

vals_m = (D[valid] * scale).astype(float)
print(
    "valid depth (m): min",
    vals_m.min(),
    "med",
    np.median(vals_m),
    "max",
    vals_m.max()
)


FFB 17
depth_scale: 0.0010000000474974513
valid %: 92.78385416666667
valid depth (m): min 0.4480000212788582 med 1.5480000735260546 max 1.6790000797482207


Object detection

In [3]:
import os, glob

# =====================================================
# MUST MATCH PREVIOUS CELLS
# =====================================================
FFB_ID = 17   # ← same FFB_ID everywhere
frames_dir = f"sample_{FFB_ID:03d}"

print("cwd:", os.getcwd())
print(f"{frames_dir} exists?", os.path.isdir(frames_dir))

rgb_files = sorted(glob.glob(f"{frames_dir}/rgb_*.png"))
print("rgb files found:", len(rgb_files))
print("first 3:", rgb_files[:3])


cwd: C:\Users\admin\Documents\Vicki\School stuff\SEGP
sample_017 exists? True
rgb files found: 67
first 3: ['sample_017\\rgb_0001.png', 'sample_017\\rgb_0002.png', 'sample_017\\rgb_0003.png']


Install Dependencies


In [4]:
# =====================================================
# DEPENDENCY CHECK / INSTALL (RUN ONCE)
# =====================================================
import sys

def ensure(pkg):
    try:
        __import__(pkg)
        print(f"[OK] {pkg} already installed")
    except ImportError:
        print(f"[INSTALLING] {pkg}")
        !{sys.executable} -m pip install {pkg}

ensure("onnxruntime")
ensure("cv2")
ensure("numpy")


[OK] onnxruntime already installed
[OK] cv2 already installed
[OK] numpy already installed


In [5]:
import onnxruntime as ort
import cv2
import numpy as np

print("ONNX Runtime version:", ort.__version__)
print("OpenCV version:", cv2.__version__)
print("NumPy version:", np.__version__)


ONNX Runtime version: 1.23.2
OpenCV version: 4.10.0
NumPy version: 2.2.6


In [6]:
# =====================================================
# LOCAL YOLOv8 ONNX DETECTOR CLASS (RUN FIRST)
# =====================================================

import cv2
import numpy as np
import onnxruntime as ort
from pathlib import Path

class LocalYOLODetector:
    def __init__(self, model_path, input_size=640, conf_threshold=0.25):
        self.input_size = input_size
        self.conf_threshold = conf_threshold

        model_path = Path(model_path)
        if not model_path.exists():
            raise FileNotFoundError(f"ONNX model not found: {model_path}")

        self.session = ort.InferenceSession(str(model_path))
        self.input_name = self.session.get_inputs()[0].name
        self.output_name = self.session.get_outputs()[0].name

        print(f"[OK] Loaded ONNX YOLO model: {model_path}")

    def _preprocess(self, image):
        img = cv2.resize(image, (self.input_size, self.input_size))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))  # HWC → CHW
        return np.expand_dims(img, axis=0)

    def detect(self, image):
        H, W = image.shape[:2]
        inp = self._preprocess(image)

        output = self.session.run([self.output_name], {self.input_name: inp})[0]
        output = np.squeeze(output)

        # YOLOv8 ONNX output: (5, N) → (N, 5)
        if output.shape[0] <= 10:
            output = output.T

        x, y, w, h, conf = output[:, 0], output[:, 1], output[:, 2], output[:, 3], output[:, 4]
        keep = conf >= self.conf_threshold
        x, y, w, h, conf = x[keep], y[keep], w[keep], h[keep], conf[keep]

        scale_x = W / self.input_size
        scale_y = H / self.input_size

        detections = []
        for i in range(len(conf)):
            x1 = int((x[i] - w[i] / 2) * scale_x)
            y1 = int((y[i] - h[i] / 2) * scale_y)
            x2 = int((x[i] + w[i] / 2) * scale_x)
            y2 = int((y[i] + h[i] / 2) * scale_y)

            x1 = max(0, min(W - 1, x1))
            y1 = max(0, min(H - 1, y1))
            x2 = max(0, min(W - 1, x2))
            y2 = max(0, min(H - 1, y2))

            if x2 <= x1 or y2 <= y1:
                continue

            detections.append({
                "bbox": [x1, y1, x2, y2],
                "confidence": float(conf[i]),
                "class": "ffb"
            })

        return detections


In [7]:
import glob

candidates = glob.glob("**/*.onnx", recursive=True)
print("Found ONNX models:")
for c in candidates:
    print(" -", c)


Found ONNX models:
 - yolo_model\odroid_h3_deployment\ffb_yolo.onnx


In [8]:
# =====================================================
# INITIALIZE LOCAL YOLO DETECTOR (RUN SECOND)
# =====================================================
YOLO_MODEL_PATH = "yolo_model/odroid_h3_deployment/ffb_yolo.onnx" # update path if needed

detector = LocalYOLODetector(
    model_path=YOLO_MODEL_PATH,
    input_size=640,
    conf_threshold=0.25
)

print("[READY] Local YOLO detector initialized")


[OK] Loaded ONNX YOLO model: yolo_model\odroid_h3_deployment\ffb_yolo.onnx
[READY] Local YOLO detector initialized


In [10]:
import cv2, glob, csv, os, time
import numpy as np

# =====================================================
# MUST MATCH ALL PREVIOUS CELLS
# =====================================================
FFB_ID = 17
frames_dir = f"sample_{FFB_ID:03d}"

# =====================================================
# LOCAL YOLO OUTPUT PATHS (SAFE, NON-DESTRUCTIVE)
# =====================================================
out_vis_dir = os.path.join(frames_dir, "detections_vis_onnx")
os.makedirs(out_vis_dir, exist_ok=True)

csv_path = os.path.join(frames_dir, "detections_onnx.csv")

print(f"Running LOCAL YOLO detection for FFB {FFB_ID}")
print("Frames dir:", frames_dir)
print("Output vis dir:", out_vis_dir)
print("CSV:", csv_path)

# =====================================================
# PARAMETERS (FINAL, STABLE)
# =====================================================
CONF_THRESHOLD = 0.15
MASK_BOTTOM_RATIO = 0.35   # completely hide bottom 45% (operator region)

# =====================================================
# PROCESS ALL FRAMES
# =====================================================
rgb_files = sorted(glob.glob(os.path.join(frames_dir, "rgb_*.png")))
print("Found frames:", len(rgb_files))

with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["frame", "x1", "y1", "x2", "y2", "conf", "class"])

    for rf in rgb_files:
        base = os.path.basename(rf)

        try:
            t0 = time.time()
            img = cv2.imread(rf)
            H, W = img.shape[:2]

            # ---------------------------------------------
            # MASK BOTTOM REGION (KEY FIX)
            # ---------------------------------------------
            masked = img.copy()
            y_cut = int(H * (1.0 - MASK_BOTTOM_RATIO))
            masked[y_cut:H, :] = 0  # black out bottom region

            # ---------------------------------------------
            # RUN YOLO ON MASKED IMAGE
            # ---------------------------------------------
            detections = detector.detect(masked)
            print(f"{base}: {len(detections)} preds in {time.time() - t0:.2f}s")

            # ---------------------------------------------
            # CONFIDENCE FILTER
            # ---------------------------------------------
            preds = [d for d in detections if d["confidence"] >= CONF_THRESHOLD]

            if not preds:
                writer.writerow([base, "", "", "", "", "", ""])
                continue

            # ---------------------------------------------
            # SELECT BOX CLOSEST TO IMAGE CENTER
            # ---------------------------------------------
            cx0, cy0 = W / 2.0, H / 2.0

            def center_dist2(p):
                x1, y1, x2, y2 = p["bbox"]
                cx = (x1 + x2) / 2
                cy = (y1 + y2) / 2
                dx, dy = cx - cx0, cy - cy0
                return dx * dx + dy * dy

            best = min(
                preds,
                key=lambda p: (center_dist2(p), -p["confidence"])
            )

            # ---------------------------------------------
            # DRAW + SAVE (ON ORIGINAL IMAGE)
            # ---------------------------------------------
            x1, y1, x2, y2 = map(int, best["bbox"])
            conf = float(best["confidence"])
            label = best.get("class", "ffb")

            vis = img.copy()
            cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(
                vis, f"{label} {conf:.2f}",
                (x1, max(0, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2
            )

            cv2.imwrite(os.path.join(out_vis_dir, base), vis)
            writer.writerow([base, x1, y1, x2, y2, round(conf, 4), label])

        except Exception as e:
            print("ERROR on", base, "->", repr(e))


Running LOCAL YOLO detection for FFB 17
Frames dir: sample_017
Output vis dir: sample_017\detections_vis_onnx
CSV: sample_017\detections_onnx.csv
Found frames: 67
rgb_0001.png: 2 preds in 0.28s
rgb_0002.png: 2 preds in 0.20s
rgb_0003.png: 0 preds in 0.21s
rgb_0004.png: 1 preds in 0.22s
rgb_0005.png: 3 preds in 0.11s
rgb_0006.png: 2 preds in 0.22s
rgb_0007.png: 2 preds in 0.20s
rgb_0008.png: 3 preds in 0.20s
rgb_0009.png: 3 preds in 0.24s
rgb_0010.png: 0 preds in 0.21s
rgb_0011.png: 0 preds in 0.20s
rgb_0012.png: 0 preds in 0.18s
rgb_0013.png: 0 preds in 0.21s
rgb_0014.png: 0 preds in 0.20s
rgb_0015.png: 0 preds in 0.21s
rgb_0016.png: 0 preds in 0.21s
rgb_0017.png: 0 preds in 0.19s
rgb_0018.png: 0 preds in 0.20s
rgb_0019.png: 0 preds in 0.22s
rgb_0020.png: 0 preds in 0.19s
rgb_0021.png: 0 preds in 0.22s
rgb_0022.png: 0 preds in 0.24s
rgb_0023.png: 0 preds in 0.21s
rgb_0024.png: 0 preds in 0.19s
rgb_0025.png: 0 preds in 0.20s
rgb_0026.png: 0 preds in 0.21s
rgb_0027.png: 0 preds in 0.21s


Masking


In [11]:
import os, csv, json, glob, cv2, numpy as np

# =====================================================
# MUST MATCH ALL PREVIOUS CELLS
# =====================================================
FFB_ID = 17
frames_dir = f"sample_{FFB_ID:03d}"

# =====================================================
# PATHS
# =====================================================
det_csv   = os.path.join(frames_dir, "detections_rf.csv")
intr_path = os.path.join(frames_dir, "intrinsics.json")

out_mask_dir = os.path.join(frames_dir, "masks")
out_vis_dir  = os.path.join(frames_dir, "masks_vis")
os.makedirs(out_mask_dir, exist_ok=True)
os.makedirs(out_vis_dir,  exist_ok=True)

print(f"Generating masks for FFB {FFB_ID}")
print("Frames dir:", frames_dir)

# =====================================================
# LOAD INTRINSICS
# =====================================================
meta  = json.load(open(intr_path))
scale = float(meta["depth_scale"])  # meters per depth unit

# =====================================================
# OPERATOR / SHIRT EXCLUSION RULE (FINAL)
# =====================================================
# Any bbox touching the bottom part of the image is NOT a fruit
SHIRT_Y_RATIO = 0.45   # bottom 45% is operator region

def is_operator_bbox(x1, y1, x2, y2, img_h):
    """
    Returns True if bbox overlaps the operator region.
    """
    y_cut = img_h * (1.0 - SHIRT_Y_RATIO)
    return y2 > y_cut   # bbox touches operator area

# =====================================================
# DEPTH-BAND MASK FUNCTION (UNCHANGED)
# =====================================================
def depth_band_mask(depth_u16, bbox, scale, band_cm=10):
    x1, y1, x2, y2 = map(int, bbox)
    h, w = depth_u16.shape

    x1 = max(0, min(w - 1, x1))
    x2 = max(0, min(w - 1, x2))
    y1 = max(0, min(h - 1, y1))
    y2 = max(0, min(h - 1, y2))

    if x2 <= x1 or y2 <= y1:
        return np.zeros_like(depth_u16, np.uint8)

    crop = depth_u16[y1:y2, x1:x2]
    z = crop[crop > 0]
    if z.size == 0:
        return np.zeros_like(depth_u16, np.uint8)

    z_med = np.median(z)
    band = int((band_cm / 100.0) / scale)

    m_roi = (
        (depth_u16 >= (z_med - band)) &
        (depth_u16 <= (z_med + band))
    ).astype(np.uint8) * 255

    mask = np.zeros_like(depth_u16, np.uint8)
    mask[y1:y2, x1:x2] = m_roi[y1:y2, x1:x2]
    return mask

# =====================================================
# MAP RGB → DEPTH
# =====================================================
depth_files = {
    os.path.basename(p): p
    for p in glob.glob(os.path.join(frames_dir, "depth_*.png"))
}

def rgb_to_depth(rgb_name):
    return depth_files.get(rgb_name.replace("rgb_", "depth_"))

# =====================================================
# GENERATE MASKS (FINAL, SAFE)
# =====================================================
with open(det_csv, newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        name = row["frame"]

        # Skip frames with no detection
        if not row["x1"]:
            continue

        x1, y1, x2, y2 = map(float, [row["x1"], row["y1"], row["x2"], row["y2"]])

        rgb_path   = os.path.join(frames_dir, name)
        depth_path = rgb_to_depth(name)

        if not depth_path or not os.path.exists(depth_path):
            continue

        # -------------------------------------------------
        # LOAD RGB (FOR GEOMETRY CHECK)
        # -------------------------------------------------
        rgb = cv2.imread(rgb_path)
        H, W = rgb.shape[:2]

        # -------------------------------------------------
        # REJECT OPERATOR DETECTIONS (KEY FIX)
        # -------------------------------------------------
        if is_operator_bbox(x1, y1, x2, y2, H):
            continue   # DO NOT generate mask or point cloud

        # -------------------------------------------------
        # GENERATE DEPTH MASK FROM DETECTION BBOX
        # -------------------------------------------------
        D = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)
        mask = depth_band_mask(D, (x1, y1, x2, y2), scale, band_cm=10)

        mask_path = os.path.join(out_mask_dir, name.replace("rgb_", "mask_"))
        cv2.imwrite(mask_path, mask)

        # -------------------------------------------------
        # VISUAL PREVIEW (UNCHANGED)
        # -------------------------------------------------
        overlay = rgb.copy()
        overlay[mask == 0] = (overlay[mask == 0] * 0.25).astype(np.uint8)

        vis_path = os.path.join(out_vis_dir, name)
        cv2.imwrite(vis_path, overlay)

print("✅ Masks saved to:", out_mask_dir)
print("✅ Visual previews saved to:", out_vis_dir)


Generating masks for FFB 17
Frames dir: sample_017
✅ Masks saved to: sample_017\masks
✅ Visual previews saved to: sample_017\masks_vis
